In [1]:
from pathlib import Path
import os
import gzip

# Folder containing the original IMDb datasets
DATA_DIR = Path.home() / "Workspace" / "DAT-153"

print("Dataset folder:", DATA_DIR)
print("Ratings file exists:", (DATA_DIR / "title.ratings.tsv.gz").exists())

Dataset folder: /home/DAVIDSON/dajeon/Workspace/DAT-153
Ratings file exists: True


In [10]:
import importlib.util

print("psycopg:", importlib.util.find_spec("psycopg") is not None)
print("psycopg2:", importlib.util.find_spec("psycopg2") is not None)
print("dotenv:", importlib.util.find_spec("dotenv") is not None)

psycopg: False
psycopg2: True
dotenv: True


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(DATA_DIR / ".env")

variables = [
    "DATABASE_URL",
    "DEV_DB_SERVER",
    "PGHOST",
    "PGPORT",
    "PGUSER",
    "PGPASSWORD"
]

for variable in variables:
    print(variable, "FOUND" if os.getenv(variable) else "NOT FOUND")

DATABASE_URL NOT FOUND
DEV_DB_SERVER FOUND
PGHOST FOUND
PGPORT FOUND
PGUSER FOUND
PGPASSWORD FOUND


In [3]:
import os
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    dbname="cinematic_test"
)

cur = conn.cursor()

cur.execute("SELECT current_database();")
print("Connected to:", cur.fetchone()[0])

Connected to: cinematic_test


In [13]:
cur.execute("SELECT COUNT(*) FROM schema_temporary.title_ratings;")
print("Rows before loading:", cur.fetchone()[0])

Rows before loading: 0


In [ ]:
import gzip

with gzip.open(DATA_DIR / "title.ratings.tsv.gz", "rt", encoding="utf-8") as file:
    next(file)  # Skip the header row

    cur.copy_expert(
        """
        COPY schema_temporary.title_ratings
        FROM STDIN
        WITH (
            FORMAT csv,
            DELIMITER E'\t',
            NULL '\\N',
            QUOTE E'\x01'
        )
        """,
        file
    )

conn.commit()

print("IMDb ratings data loaded successfully!")

IMDb ratings data loaded successfully!


In [15]:
cur.execute("SELECT COUNT(*) FROM schema_temporary.title_ratings;")
print("Rows after loading:", cur.fetchone()[0])

cur.execute("SELECT * FROM schema_temporary.title_ratings LIMIT 5;")

for row in cur.fetchall():
    print(row)

Rows after loading: 1726786
('tt0000001', '5.7', '2237')
('tt0000002', '5.4', '327')
('tt0000003', '6.4', '2401')
('tt0000004', '5.0', '203')
('tt0000005', '6.2', '3105')


In [ ]:
import gzip

def load_imdb(filename, table_name):
    print(f"Loading {filename} into {table_name}...")

    with gzip.open(DATA_DIR / filename, "rt", encoding="utf-8") as file:
        next(file)  # Skip column names

        cur.copy_expert(
            f"""
            COPY schema_temporary.{table_name}
            FROM STDIN
            WITH (
                FORMAT csv,
                DELIMITER E'\\t',
                NULL '\\N',
                QUOTE E'\\x01'
            )
            """,
            file
        )

    conn.commit()
    print(f"Finished loading {table_name}!")

In [17]:
files = [
    ("name.basics.tsv.gz", "name_basics"),
    ("title.akas.tsv.gz", "title_akas"),
    ("title.basics.tsv.gz", "title_basics"),
    ("title.crew.tsv.gz", "title_crew"),
    ("title.episode.tsv.gz", "title_episode"),
    ("title.principals.tsv.gz", "title_principals"),
]

for filename, table_name in files:
    try:
        load_imdb(filename, table_name)
    except Exception as e:
        conn.rollback()
        print(f"Failed to load {table_name}: {e}")
        break

Loading name.basics.tsv.gz into name_basics...
Finished loading name_basics!
Loading title.akas.tsv.gz into title_akas...
Finished loading title_akas!
Loading title.basics.tsv.gz into title_basics...
Finished loading title_basics!
Loading title.crew.tsv.gz into title_crew...
Finished loading title_crew!
Loading title.episode.tsv.gz into title_episode...
Finished loading title_episode!
Loading title.principals.tsv.gz into title_principals...
Finished loading title_principals!


In [18]:
tables = [
    "name_basics",
    "title_akas",
    "title_basics",
    "title_crew",
    "title_episode",
    "title_principals",
    "title_ratings",
]

for table in tables:
    cur.execute(f"SELECT COUNT(*) FROM schema_temporary.{table}")
    count = cur.fetchone()[0]
    print(f"{table}: {count:,} rows")

name_basics: 15,774,367 rows
title_akas: 60,003,988 rows
title_basics: 12,910,620 rows
title_crew: 12,907,853 rows
title_episode: 9,985,716 rows
title_principals: 102,590,405 rows
title_ratings: 1,726,786 rows


In [5]:
cur.execute("SELECT COUNT(*) FROM schema_temporary.title_ratings;")
print("Ratings rows:", cur.fetchone()[0])

Ratings rows: 1726786
